***
***
# Tutorial illustraing the use of C3S for DFTB Repulsive Potential Parametrization
***
***
<br>
In this notebook we perform repulsive potential parametrization for Densisty Functional Tight Binding (DFTB). 

<br><br><br><br><br>

***
***

## 1. Import modules
Import the modules needed to generate the training data.

In [ ]:
from ccs_fit.scripts.ccs_build_db import ccs_build_db as ccs_build_db
from ccs_fit.fitting.main import twp_fit as ccs_fit
from ccs_fit.ase_calculator.ccs_ase_calculator import spline_table
from ccs_fit.scripts.ccs_export_sktable import ccs_export_sktable

import numpy as np
import matplotlib.pyplot as plt
import json

***
***
## 2. Build ASE database(s)
CCS Training-set are conveniently collected in the from of ASE databases. In the case of DFTB repulsive fitting we make two separate databases, one with the DFT-data and one with corresponding DFTB-data. The database(s) will contain forces, energies and stresses. A list (`file_list`) specifying the folders where the DFT (and DFTB) outputs are stored should be provided. Each line of 'file_list' should contain two columns, the first giving the location of the DFT data and the second the corresponding DFTB data. The DFTB data has to be in the form of a `results.tag` file.

The `file_list` looks like this:

```
S1/VASP/out.xyz   S1/DFTB/results.tag
S2/VASP/out.xyz   S2/DFTB/results.tag
S3/VASP/out.xyz   S3/DFTB/results.tag
```

In [ ]:
 
file_list ='''S1/VASP/out.xyz   S1/DFTB/results.tag
S2/VASP/out.xyz   S2/DFTB/results.tag
S3/VASP/out.xyz   S3/DFTB/results.tag'''

with open('trainset.list', 'w') as f:
    f.write(file_list)

ccs_build_db(mode='DFTB', DFT_DB='DFT.db', DFTB_DB='DFTB.db',file_list='trainset.list',overwrite=True)

***
***
## 3. Inspecting the databases
The database(s) can be inspected using the 'ase db' and ase gui' commands of ASE.

In [ ]:
%%capture
!ase gui DFT.db

***
***
## 4. Fitting the repulsive potential

In order to fit repulsive potentials we have to point to the databases with the DFT and DFTB data. 

If the `Twobody` is not specified, all unique atoms pairs in the data-set will get a spline potetnial using a cutoff of 5 Å, repulsive constraints and a resolution of 0.1 Å. Defaults can be overridden by including a custom `Twobody` block, see basic tutorial for more details.  

In [ ]:
input={
        "General": {
                "Interface": "DFTB",
        },
        "TrainSet": "DFT.db", 
        "TrainSetDFTB": "DFTB.db"
}

#SAVE TO FILE
with open('CCS_input.json', 'w') as f:
    json.dump(input, f, indent=8)

#RUN FIT
ccs_fit("CCS_input.json")

***
***

## 5. Inspect results

Note that this is a very minimalsitic example with merly 3 structres. For reliable repulsive potentials we would typically need more.

In [ ]:
error=np.loadtxt("CCS_error_energies.out")

#MAKE THE PLOT
EDFT=error[:,0]/ error[:,3]
EDFTB=error[:,1]/ error[:,3]
x_line=[min(EDFT)-0.25, max(EDFT)+0.25]
plt.plot(x_line,x_line,'-')
plt.xlim(x_line)
plt.ylim(x_line)
plt.xlabel('Target repulsive energy (eV)')
plt.ylabel('Predicted repulsive energy (eV)')
plt.scatter(EDFT,EDFTB)
plt.show()


with open("CCS_params.json", "r") as f:
    CCS_params = json.load(f)

for pair in CCS_params["Two_body"]:
    
    r=np.arange(CCS_params["Two_body"][pair]["r_min"], CCS_params["Two_body"][pair]["r_cut"], 0.01)
    elem1, elem2 = pair.split("-")
    tb = spline_table(elem1, elem2, CCS_params)
    y = [tb.eval_energy(rs) for rs in r]
    plt.plot(r,y,'-',color='black',label=f"{pair} repulsive")
    plt.xlabel('Distance (Å)')
    plt.ylabel('Energy (eV)')
    plt.legend()
    plt.show()

***
***

##  6. Export spline-table

With the `ccs_export_sktable` command we can export our fitted repulsive potentials into `DFTB+` Slater-Koster format. The Files will be named according to elements in the pair separated with `-` and have suffix `.spl`. In this example that means `Si-Si.spl`.

In [ ]:
ccs_export_sktable("CCS_params.json")

***
***

##  7. Reset notebook

In [ ]:
import os
import glob
import ipywidgets as widgets
from IPython.display import display, clear_output

# Define the directories
current_directory = os.getcwd()

# Function to remove files except for the specified file
def remove_files_except(directory, exception_file):
    for file_path in glob.glob(os.path.join(directory, '*')):
        if os.path.isfile(file_path) and not file_path.endswith(exception_file):
            os.remove(file_path)

# Cleanup function triggered by button click
def cleanup(b):
    with out:
        clear_output(wait=True)
        remove_files_except(current_directory, 'ipynb')
        print("Cleanup completed.")

# Function to cancel cleanup
def cancel_cleanup(b):
    with out:
        clear_output(wait=True)
        print("No changes made.")

# Create Yes/No buttons
button_yes = widgets.Button(description="Yes", button_style='danger')
button_no = widgets.Button(description="No", button_style='success')

button_yes.on_click(cleanup)
button_no.on_click(cancel_cleanup)

# Display prompt
print("Would you like to clean up?")
display(button_yes, button_no)

# Output area for messages
out = widgets.Output()
display(out)